# Cosmic Census
## Celestial Object Classification with Big Data

### About this project
#### Sloan Digital Sky Survey (SDSS) | PySpark | Pandas | Plotly | TensorFlow

The **Sloan Digital Sky Survey (SDSS)** is one of the largest astronomical surveys in history,
having catalogued over 500 million celestial objects since the year 2000.
This project uses 100,000 real SDSS observations to build a **complete Big Data pipeline**
that answers the following scientific question:

> *"Given a set of photometric measurements of an object in the sky, is it a star, a galaxy, or a quasar (QSO)?"*

### Astronomical domain glossary

| Term | Meaning |
|---|---|
| **Galaxy** | System of billions of stars bound by gravity (e.g., the Milky Way) |
| **Star** | Celestial body undergoing nuclear fusion; the ones in this dataset are within our own galaxy |
| **Quasar (QSO)** | "Quasi-Stellar Object" — an extremely luminous and distant active galactic nucleus |
| **u, g, r, i, z** | The 5 SDSS photometric filters: u=ultraviolet, g=green, r=red, i=near-infrared, z=infrared. The telescope photographs each object 5 times (one per filter), producing 5 brightness values (magnitudes) |
| **Magnitude** | Measure of an object's brightness: the higher the number, the fainter the object |
| **Redshift (z)** | Red shift of light — indicates distance. Stars: z ≈ 0 (nearby). Galaxies: z ~ 0.5. Quasars: z > 1 (the most distant objects ever observed) |
| **Color index** | Difference between magnitudes in two filters (e.g., u−g). Different object types occupy distinct regions in color space — it is their spectral "fingerprint" |

### What is Machine Learning here?

The algorithm is **not programmed with rules**. Instead, you show it thousands of examples
already classified by astronomers (stars, galaxies, quasars) and the algorithm discovers
on its own which brightness and color patterns separate the three classes.
This is called **supervised learning**.

**Random Forest** is one of the most widely used algorithms: it builds hundreds of "decision trees",
each trained on a random subset of the data, and takes a majority vote at the end.
The "forest" of trees is far more robust than a single tree.

### Curriculum topics covered

| Module | What we applied |
|---|---|
| Big Data principles | The 5 Vs, ecosystem, data lake |
| Hadoop & storage | HDFS vs RDBMS, Parquet as Data Lake |
| PySpark | SparkSession, RDDs, DataFrames, MapReduce, transformations |
| Pandas + Plotly | Full EDA, 5 interactive visualizations, sky map |
| Big Data Analytics | KDD pipeline, Scikit-learn Random Forest, neural network with TensorFlow/Keras |

### Pipeline architecture

```
[SDSS CSV]  ->  [Parquet Data Lake]  ->  [PySpark: cleaning + features]
    ->  [Pandas: EDA + Plotly]  ->  [TensorFlow: classification]
    ->  [Evaluation + Spark MLlib Pipeline]
```


---
## 0. Install dependencies

Run this cell once.
- **Google Colab**: everything is pre-installed except `pyspark`
- **VSCode / local Jupyter**: requires **Java 8+** on your PATH (needed by Spark)


In [ ]:
import subprocess, sys

packages = [
    "pyspark",       # Apache Spark for Python
    "pandas",        # Data analysis
    "numpy",         # Numerical operations
    "plotly",        # Interactive visualizations
    "scikit-learn",  # Classic Machine Learning
    "tensorflow",    # Deep Learning / Neural networks
    "requests",      # Dataset download
    "pyarrow",       # Parquet support (Data Lake)
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])

print("All dependencies installed!")


---
## 1. Imports and global configuration

Best practice: centralize all imports in a single cell at the top of the notebook.


In [ ]:
import os, warnings, requests, time, pickle, json
from datetime import datetime
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
from pyspark.ml.classification import RandomForestClassifier as SparkRFC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Classic Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import classification_report, confusion_matrix

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"PySpark    {__import__('pyspark').__version__}")
print(f"Pandas     {pd.__version__}")
print(f"TensorFlow {tf.__version__}")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Astronomical color palette
COLORS = {"GALAXY": "#5B8FD4", "STAR": "#F5C842", "QSO": "#E05C5C"}


---
## 2. Big Data principles — context and data acquisition

### The 5 Vs of Big Data applied to SDSS

| V | In SDSS |
|---|---|
| **Volume** | Over 500 million objects, ~60 TB of raw images |
| **Velocity** | New observations every observing night |
| **Variety** | Photometric (u,g,r,i,z), spectroscopic, and FITS image data |
| **Veracity** | Measurements with calibrated instrumental uncertainty |
| **Value** | Discovery of quasars, distant galaxies, dark matter |

### Data Lake concept

We ingest raw data and store it in **Parquet** (compressed columnar format).
In production this file would sit on **HDFS** or Amazon S3. Here we simulate
the same architectural pattern locally.

```
[Source: SDSS CSV]  ->  [Data Lake: ./data/sdss_lake.parquet]
                                  |
                    +-------------+
                    |   PySpark   |   distributed processing
                    +-------------+
```


In [ ]:
import os, requests
import pandas as pd

os.makedirs("data", exist_ok=True)
CSV     = "data/sdss_raw.csv"
PARQUET = "data/sdss_lake.parquet"

if not os.path.exists(CSV):
    print("Downloading SDSS DR17 dataset...")
    url = "https://raw.githubusercontent.com/SayamAlt/Stellar-Classification---Sloan-Digital-Sky-Survey-17/main/star_classification.csv"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(CSV, "wb") as f:
        f.write(r.content)
    print(f"Download complete: {len(r.content)/1e6:.1f} MB")
else:
    print("Dataset already cached.")

df_raw = pd.read_csv(CSV)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)


---
## 3. Hadoop and Data Lake — storing as Parquet

### HDFS vs local filesystem (in this project)

In production, Parquet files would live on **HDFS** distributed across multiple cluster nodes.
Here we simulate the same architectural pattern using the local filesystem.

**Why Parquet instead of CSV?**

| Feature | CSV | Parquet |
|---|---|---|
| Layout | Row-oriented | Column-oriented |
| Compression | None | Snappy, Gzip, LZ4 |
| Schema | Inferred on every read | Embedded in the file |
| Read speed | Slow | Very fast |
| Ecosystem compatibility | Universal | Hadoop, Spark, Hive, Presto |


In [ ]:
if not os.path.exists(PARQUET):
    df_raw.to_parquet(PARQUET, index=False, compression="snappy")
    tc = os.path.getsize(CSV) / 1e6
    tp = os.path.getsize(PARQUET) / 1e6
    print(f"Data Lake created!")
    print(f"  CSV     : {tc:.2f} MB")
    print(f"  Parquet : {tp:.2f} MB  ({(1-tp/tc)*100:.0f}% smaller)")
else:
    print("Data Lake already exists.")

print("""
  HDFS / Data Lake          |   RDBMS (relational database)
  --------------------------|-----------------------------
  Schema-on-read            |   Schema-on-write
  Accepts raw data          |   Data must be structured
  Horizontal scaling        |   Vertical scaling
  Low cost per TB           |   High cost per TB
  Ideal for Big Data        |   Ideal for ACID transactions
""")


---
## 4. Apache Spark — initialization and core concepts

### Spark architecture (simplified)

```
Driver Program (this notebook)
       |
       v
 SparkContext / SparkSession
       |
  [Cluster Manager]  <-- production: YARN, Kubernetes, Mesos
       |               here: local mode (simulates a cluster)
  [Executors / Workers]  process data in parallel
```

**RDD vs DataFrame**

- **RDD** (Resilient Distributed Dataset): low-level abstraction, immutable, fault-tolerant
- **DataFrame**: high-level abstraction over RDDs with the Catalyst optimizer — always prefer this


In [ ]:
spark = (
    SparkSession.builder
    .appName("CosmicCensus")
    .master("local[*]")                        # uses all available cores
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession started!")
print(f"  Version : {spark.version}")
print(f"  Master  : {spark.sparkContext.master}")
print(f"  Cores   : {spark.sparkContext.defaultParallelism}")
print("  Spark UI: http://localhost:4040  (available during execution)")


---
## 5. PySpark — loading, exploration, and MapReduce

We demonstrate **two ways** to process data with Spark:

1. **DataFrame API** — high-level, automatically optimized by Catalyst
2. **RDD + MapReduce** — low-level, shows the fundamental Hadoop/Spark mechanism


In [ ]:
sdf = spark.read.parquet(PARQUET)

print("Spark DataFrame schema:")
sdf.printSchema()
print(f"Total records : {sdf.count():,}")
print(f"Partitions    : {sdf.rdd.getNumPartitions()}")
sdf.show(5, truncate=False)


In [ ]:
# Register as a temporary SQL view
sdf.createOrReplaceTempView("sdss")

print("Class distribution via Spark SQL:")
spark.sql("""
    SELECT class,
           COUNT(*) AS total,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM sdss
    GROUP BY class
    ORDER BY total DESC
""").show()

features_photo = [c for c in ["u","g","r","i","z","redshift"] if c in sdf.columns]
print("Descriptive statistics:")
sdf.select(features_photo).describe().show()


In [ ]:
# MapReduce with PySpark RDD
# Fundamental pattern: Map --> Shuffle --> Reduce
# The same mechanism behind Hadoop MapReduce,
# but executed in RAM — that's why Spark is ~100x faster.

rdd_base = sdf.select("class", "redshift").rdd

# MAP: each row becomes (class, (redshift, 1))
rdd_map = rdd_base.map(lambda row: (row["class"], (row["redshift"], 1)))

# REDUCE: sum redshifts and counts per class
rdd_red = rdd_map.reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))

# FINAL MAP: compute mean redshift
rdd_res = rdd_red.map(
    lambda kv: (kv[0], kv[1][1], round(kv[1][0]/kv[1][1], 4))
).sortBy(lambda x: -x[1])

print("Class        | Total      | Mean redshift")
print("-" * 45)
for cls, tot, z in rdd_res.collect():
    print(f"{cls:<12} | {tot:>10,} | {z:>14.4f}")

print("""
What happened:
  MAP    -> transforms each row into a (key, value) pair
  SHUFFLE-> groups pairs by the same key (across the cluster, over the network)
  REDUCE -> combines values of the same key into a single result
""")


---
## 6. PySpark — transformations and feature engineering

Spark operates **lazily**: transformations only define the **DAG** (Directed Acyclic Graph)
and do not execute immediately. Actual execution only happens when an **action** is called
(`.count()`, `.show()`, `.collect()`).

### Color indices — the spectral "fingerprint"

The u, g, r, i, z filters measure an object's brightness at 5 wavelengths.
The **difference between two adjacent filters** — called a color index — is far more
informative than each raw magnitude on its own:

| Index | Filters | What it indicates |
|---|---|---|
| **u − g** | Ultraviolet − Green | UV emission (quasars and young stars have lower values) |
| **g − r** | Green − Red | Effective surface temperature |
| **r − i** | Red − Near-infrared | Reddening by interstellar dust |
| **i − z** | Near-infrared − z | Presence of cold molecules (M-type stars) |

Galaxies, stars, and quasars occupy completely different regions in this color space
— that is what the color diagram (chart 2) will show visually.


In [ ]:
features_photo = [c for c in ["u","g","r","i","z","redshift"] if c in sdf.columns]

# 1. Remove duplicates and nulls
sdf_clean = sdf.dropDuplicates().dropna(subset=features_photo + ["class"])

# 2. Filter physically impossible values (instrumental noise)
sdf_clean = sdf_clean.filter(
    (F.col("redshift") >= -0.01) &
    (F.col("u") > 0) & (F.col("g") > 0) & (F.col("r") > 0)
)

# 3. Feature engineering: photometric color indices
#    Color = magnitude difference between adjacent bands
#    Each object type occupies a distinct region in color space
sdf_feat = (
    sdf_clean
    .withColumn("color_ug", F.col("u") - F.col("g"))  # ultraviolet - green
    .withColumn("color_gr", F.col("g") - F.col("r"))  # green - red
    .withColumn("color_ri", F.col("r") - F.col("i"))  # red - near-infrared
    .withColumn("color_iz", F.col("i") - F.col("z"))  # near-infrared - z
    # 4. Encode the class as an integer (required by ML algorithms)
    .withColumn("label",
        F.when(F.col("class") == "GALAXY", 0)
         .when(F.col("class") == "STAR",   1)
         .otherwise(2)
    )
)

n_final = sdf_feat.count()
print(f"Dataset after cleaning: {n_final:,} records")
print(f"Removed               : {sdf.count() - n_final:,}")
sdf_feat.select("class","label","color_ug","color_gr","color_ri","color_iz").show(4)


---
## 7. Exploratory analysis with Pandas

After distributed processing in Spark, we convert to Pandas for richer analysis.
This is the real-world industry pattern: **Spark for scale, Pandas for insight**.

> **Golden rule:** call `.toPandas()` only after filtering and aggregating with Spark.
> Never collect a DataFrame with billions of rows to the driver!


In [ ]:
columns = features_photo + ["color_ug","color_gr","color_ri","color_iz","class","label"]
for col in ["alpha", "delta"]:
    if col in sdf_feat.columns:
        columns.append(col)

df = sdf_feat.select(columns).toPandas()

print(f"Pandas DataFrame: {df.shape}")
print("\nClass distribution:")
print(df["class"].value_counts())
print("\nNull values per column:")
print(df.isnull().sum())
df.describe().round(3)


---
## 8. Visualizations with Plotly — exploring the universe

**Why Plotly?**
- Interactive charts: zoom, hover with real data, region selection
- Exportable as HTML (runs in any browser without a server)
- Industry standard for scientific dashboards

### Feature legend used in the charts

| Feature | Type | Description |
|---|---|---|
| **u** | Magnitude | Brightness in the ultraviolet filter (~354 nm) |
| **g** | Magnitude | Brightness in the green filter (~477 nm) |
| **r** | Magnitude | Brightness in the red filter (~623 nm) |
| **i** | Magnitude | Brightness in the near-infrared filter (~762 nm) |
| **z** | Magnitude | Brightness in the z filter (~913 nm) |
| **redshift** | Physics | Red shift — distance indicator |
| **color_ug** | Color | u − g: the bluer/more UV the object, the lower the value |
| **color_gr** | Color | g − r: approximate temperature of the object |
| **color_ri** | Color | r − i: reddening and spectral type |
| **color_iz** | Color | i − z: near-infrared characteristics |


In [ ]:
# Chart 1 — Class distribution
counts = df["class"].value_counts().reset_index()
counts.columns = ["Class", "Total"]

fig1 = px.bar(
    counts, x="Class", y="Total", color="Class",
    color_discrete_map=COLORS,
    title="Distribution of celestial objects in the SDSS catalogue",
    labels={"Total": "Number of objects"},
    text="Total"
)
fig1.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig1.update_layout(showlegend=False, height=400, title_font_size=16)
fig1.show()
print("Galaxies dominate the catalogue — the visible universe is mostly large-scale structure.")


In [ ]:
# Chart 2 — Color diagram (photometric equivalent of the HR diagram)
#
# The HR (Hertzsprung-Russell) diagram is the most famous chart in astronomy:
# it plots temperature vs. luminosity and reveals the structure of stellar populations.
# The photometric color diagram does the same with SDSS data:
# each object type occupies a different region in u-g vs g-r space.
#
# X axis (g-r): bluer objects are on the left, redder on the right
# Y axis (u-g): objects with strong UV emission are at the bottom

df_sample = (
    df.groupby("class")
      .apply(lambda g: g.sample(min(5000, len(g)), random_state=SEED))
      .reset_index(drop=True)
)

fig2 = px.scatter(
    df_sample, x="color_gr", y="color_ug",
    color="class", color_discrete_map=COLORS,
    opacity=0.4,
    title="SDSS Color Diagram (u-g vs g-r) — each object type occupies a distinct region",
    labels={"color_gr": "Color g−r  (blue ← → red)", "color_ug": "Color u−g  (strong UV ↓)", "class": "Class"},
    hover_data={"redshift": ":.3f"},
)
fig2.update_traces(marker=dict(size=3))
fig2.update_layout(height=500, title_font_size=15)
fig2.show()
print("Galaxies and quasars: lower u-g (bluer UV) due to emission from active nuclei.")
print("Stars: well-defined main sequence — temperature decreases from left to right.")
print("This visual separation is exactly what the ML model will learn to recognize!")


In [ ]:
# Chart 3 — Redshift distribution by class
# Redshift is the cosmic distance indicator: higher z means farther away.
fig3 = px.histogram(
    df_sample, x="redshift", color="class",
    color_discrete_map=COLORS,
    nbins=80, barmode="overlay", opacity=0.65,
    title="Redshift distribution by celestial object type",
    labels={"redshift": "Redshift (z)", "count": "Frequency"},
    range_x=[-0.01, 3.5]
)
fig3.update_layout(height=420, title_font_size=16)
fig3.show()
print("Stars   : redshift ~ 0  (Milky Way, negligible distances)")
print("Galaxies: up to z ~ 0.5  (billions of light-years away)")
print("Quasars : reach z > 3  (the most distant objects ever observed!)")


In [ ]:
# Chart 4 — Sky map (SDSS coordinates)
if "alpha" in df.columns and "delta" in df.columns:
    df_map = (
        df.groupby("class")
          .apply(lambda g: g.sample(min(2000, len(g)), random_state=SEED))
          .reset_index(drop=True)
    )
    fig4 = px.scatter(
        df_map, x="alpha", y="delta",
        color="class", color_discrete_map=COLORS,
        opacity=0.5,
        title="Sky map — position of SDSS objects in the sky (RA × Dec)",
        labels={"alpha": "Right Ascension (degrees)", "delta": "Declination (degrees)", "class": "Class"},
        hover_data={"redshift": ":.4f"}
    )
    fig4.update_traces(marker=dict(size=2))
    fig4.update_layout(
        height=450, title_font_size=16,
        plot_bgcolor="#0a0a1a", paper_bgcolor="#0a0a1a", font_color="white"
    )
    fig4.show()
    print("SDSS covers ~1/3 of the sky — that is why the distribution is not uniform.")
else:
    print("Coordinate columns not present — skipping sky map.")


In [ ]:
# Chart 5 — Feature correlation matrix
#
# Correlation measures how much two variables "move together":
#   +1.0 = perfect positive correlation (one goes up, the other goes up)
#    0.0 = no relationship
#   -1.0 = perfect negative correlation (one goes up, the other goes down)
#
# For Machine Learning, highly correlated features are redundant —
# the model gains no new information from them.
# That is why we created the color indices: they capture orthogonal (independent) information.

feat_corr = [f for f in
    ["u","g","r","i","z","redshift","color_ug","color_gr","color_ri","color_iz"]
    if f in df.columns]

corr = df[feat_corr].corr().round(2)
fig5 = px.imshow(
    corr, text_auto=True,
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Feature correlation — raw magnitudes vs. color indices",
    aspect="auto"
)
fig5.update_layout(height=480, title_font_size=15)
fig5.show()
print("u, g, r, i, z: highly correlated (dark blue) — redundant information.")
print("color_*: lower correlations — each index captures a different spectral aspect.")
print("Conclusion: color indices are richer features for the classifier!")


---
## 9. KDD Pipeline — preparing for Machine Learning

### The KDD process (Knowledge Discovery in Databases)

```
Raw data  ->  Selection  ->  Pre-processing  ->  Transformation
    ->  Mining (ML algorithm)  ->  Interpretation  ->  Knowledge
```

We have already completed the first 4 stages with Spark and Pandas. Now we apply the algorithms.


In [ ]:
FEATURES = [f for f in
    ["u","g","r","i","z","redshift","color_ug","color_gr","color_ri","color_iz"]
    if f in df.columns]

X = df[FEATURES].values
y = df["label"].values

# Normalization (mean 0, std 1) — essential for neural networks
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Stratified 80/20 train/test split
# stratify=y guarantees the same class proportion in both sets
X_tr, X_te, y_tr, y_te = train_test_split(
    X_sc, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Features : {FEATURES}")
print(f"Train    : {X_tr.shape[0]:,} samples")
print(f"Test     : {X_te.shape[0]:,} samples")
print("\nClass distribution in training set:")
for cls, name in [(0,"GALAXY"),(1,"STAR"),(2,"QSO")]:
    n = (y_tr == cls).sum()
    print(f"  {name:<8}: {n:>6,} ({n/len(y_tr)*100:.1f}%)")


---
## 10. Machine Learning with Scikit-Learn — baseline

### What is Random Forest?

Imagine you want to decide whether a celestial object is a galaxy, star, or quasar.
You could create a simple rule: *"if redshift > 1, it is probably a quasar"*.
That is a **decision tree** — a sequence of questions about the features.

The problem is that a single tree tends to memorize the training data (overfitting).
**Random Forest** solves this by building 100+ trees, each trained on a different
random subset of the data and features.
When classifying a new object, all trees vote and the most-voted class wins.

> "The wisdom of the crowd" applied to algorithms.

### Why use it as a baseline?

Before training a neural network (more complex and slower), we check whether a simpler
model already solves the problem well. If Random Forest already reaches 95%+,
the neural network will only be useful if it can improve significantly.

### Features available to the model

| Feature | Description | Why it helps |
|---|---|---|
| u, g, r, i, z | Magnitudes in the 5 filters | Absolute brightness of the object |
| redshift | Red shift | Direct distance indicator |
| color_ug | u − g | UV emission — quasars have low values |
| color_gr | g − r | Effective temperature |
| color_ri | r − i | Interstellar reddening |
| color_iz | i − z | Near-infrared characteristics |


In [ ]:
print("Training Random Forest...")
t0 = time.time()

rf = RFC(n_estimators=100, max_depth=15, n_jobs=-1, random_state=SEED)
rf.fit(X_tr, y_tr)

t1 = time.time()
y_pred_rf = rf.predict(X_te)
acc_rf = (y_pred_rf == y_te).mean()

print(f"Training time: {t1-t0:.1f}s")
print(f"Test accuracy: {acc_rf*100:.2f}%")
print("\nClassification report:")
print(classification_report(y_te, y_pred_rf, target_names=["GALAXY","STAR","QSO"]))

# Feature importance
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig_i = px.bar(
    imp.reset_index(), x=0, y="index", orientation="h",
    title="Feature importance (Random Forest)",
    labels={"index": "Feature", 0: "Importance"},
    color=0, color_continuous_scale="Blues"
)
fig_i.update_layout(height=400, showlegend=False, title_font_size=16)
fig_i.show()
print("Redshift is the most important feature: stars have z~0, quasars have z >> 0.")


---
## 11. Deep Learning with TensorFlow/Keras — neural network classifier

### Why a neural network after Random Forest?

Random Forest learns linear combinations of features.
A neural network learns **non-linear** combinations — potentially capturing
subtler patterns in the photometric color space.

In practice, for tabular data like this, both techniques perform similarly.
Comparing the two is good scientific practice and demonstrates mastery of both approaches.

### Architecture

```
Input (10 features: u, g, r, i, z, redshift, color_ug, color_gr, color_ri, color_iz)
       |
  Dense(256, ReLU) + BatchNorm + Dropout(0.3)
       |
  Dense(128, ReLU) + BatchNorm + Dropout(0.2)
       |
  Dense(64,  ReLU) + BatchNorm
       |
  Dense(3, Softmax)   <-- GALAXY, STAR, QSO
```

**Batch Normalization**: normalizes internal activations at each layer — more stable training.
**Dropout**: randomly disables neurons during training — prevents memorizing the data (overfitting).
**Softmax**: converts final values into probabilities that sum to 1.0 — ideal for classification.


In [ ]:
n_feat = X_tr.shape[1]

model = Sequential([
    Dense(256, input_shape=(n_feat,), activation="relu", name="dense_1"),
    BatchNormalization(name="bn_1"),    # normalizes activations -> more stable training
    Dropout(0.3, name="drop_1"),        # disables 30% of neurons -> prevents overfitting

    Dense(128, activation="relu", name="dense_2"),
    BatchNormalization(name="bn_2"),
    Dropout(0.2, name="drop_2"),

    Dense(64, activation="relu", name="dense_3"),
    BatchNormalization(name="bn_3"),

    # Softmax: converts logits into probabilities that sum to 1.0
    Dense(3, activation="softmax", name="output"),
], name="CosmicCensusNet")

model.summary()


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",  # integer labels, not one-hot
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1)
]

print("Training neural network...")
t0 = time.time()

history = model.fit(
    X_tr, y_tr,
    epochs=60, batch_size=512,
    validation_split=0.15,
    callbacks=callbacks, verbose=1
)

t1 = time.time()
print(f"\nTraining: {t1-t0:.1f}s  ({len(history.history['loss'])} epochs)")


In [ ]:
# Learning curves
h = history.history
ep = list(range(1, len(h["loss"])+1))

fig_h = make_subplots(rows=1, cols=2, subplot_titles=["Loss", "Accuracy"])
for pfx, name in [("","Train"),("val_","Validation")]:
    fig_h.add_trace(go.Scatter(x=ep, y=h[pfx+"loss"],
                               name=f"Loss {name}", mode="lines"), row=1, col=1)
    fig_h.add_trace(go.Scatter(x=ep, y=h[pfx+"accuracy"],
                               name=f"Acc {name}", mode="lines"), row=1, col=2)
fig_h.update_layout(height=380, title_text="Learning curves", title_font_size=16)
fig_h.show()

loss_te, acc_te = model.evaluate(X_te, y_te, verbose=0)
y_pred_nn = np.argmax(model.predict(X_te, verbose=0), axis=1)

print(f"\nResults on TEST set:")
print(f"  Loss     : {loss_te:.4f}")
print(f"  Accuracy : {acc_te*100:.2f}%")
print("\nClassification report:")
print(classification_report(y_te, y_pred_nn, target_names=["GALAXY","STAR","QSO"]))


In [ ]:
# Interactive confusion matrix
cm = confusion_matrix(y_te, y_pred_nn)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
names = ["GALAXY", "STAR", "QSO"]

fig_cm = px.imshow(
    cm_pct, x=names, y=names,
    text_auto=".1f",
    color_continuous_scale="Blues",
    labels={"x": "Predicted", "y": "Actual", "color": "% row"},
    title="Confusion matrix — Neural Network (% per actual class)"
)
fig_cm.update_layout(height=420, title_font_size=16)
fig_cm.show()

print("How to read: main diagonal = correct predictions | off-diagonal = misclassifications")
print(f"\nModel comparison:")
print(f"  Random Forest  : {(y_pred_rf == y_te).mean()*100:.2f}%")
print(f"  Neural Network : {acc_te*100:.2f}%")


---
## 12. PySpark ML Pipeline — scaling to real Big Data

Scikit-Learn and TensorFlow run on a single machine. For **real terabyte-scale datasets**
we use **Spark MLlib**, which distributes training across the entire cluster with automatic fault tolerance.


In [ ]:
feat_spark = [f for f in FEATURES if f in sdf_feat.columns]

# VectorAssembler: combines columns into a single vector (required by Spark ML)
asm = VectorAssembler(inputCols=feat_spark, outputCol="features_raw")

# Distributed StandardScaler
scl = SparkScaler(inputCol="features_raw", outputCol="features",
                   withStd=True, withMean=True)

# Distributed Random Forest
rf_sp = SparkRFC(featuresCol="features", labelCol="label",
                  numTrees=50, maxDepth=10, seed=SEED)

# Pipeline: chains the stages (same concept as sklearn Pipeline)
pipeline = Pipeline(stages=[asm, scl, rf_sp])

sdf_ml = sdf_feat.select(feat_spark + ["label"]).dropna()
sdf_tr, sdf_te = sdf_ml.randomSplit([0.8, 0.2], seed=SEED)

print(f"Spark train: {sdf_tr.count():,} | Test: {sdf_te.count():,}")
print("Training Spark Pipeline...")
t0 = time.time()
mdl_sp = pipeline.fit(sdf_tr)
t1 = time.time()

preds = mdl_sp.transform(sdf_te)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
acc_sp = evaluator.evaluate(preds)

print(f"Training time: {t1-t0:.1f}s")
print(f"Accuracy (Spark MLlib): {acc_sp*100:.2f}%")
print("""
On a real cluster this pipeline would:
  - Process data in parallel across all machines
  - Handle datasets larger than a single machine's RAM
  - Tolerate failures: if a node goes down, Spark recomputes automatically
  - Scale to terabytes without changing a single line of code
""")


---
## 13. Prediction function — using the model in production

We wrap the model in a clean, documented function ready to be integrated
into a REST API (FastAPI, Flask) or inference pipeline.


In [ ]:
CLASS_NAMES = {0: "GALAXY", 1: "STAR", 2: "QUASAR"}
CLASS_DESC  = {
    0: "Galaxy: gravitationally bound system of billions of stars",
    1: "Star: nuclear fusion body, typically within the Milky Way",
    2: "Quasar (QSO): extremely luminous and distant active galactic nucleus",
}

def classify_celestial_object(u, g, r, i, z, redshift):
    """
    Classify a SDSS celestial object using the trained neural network.

    Parameters: u, g, r, i, z (photometric magnitudes), redshift
    Returns   : dict with class, confidence, and probabilities
    """
    input_data = np.array([[u, g, r, i, z, redshift,
                              u-g, g-r, r-i, i-z]])
    input_norm = scaler.transform(input_data[:, :len(FEATURES)])
    probs = model.predict(input_norm, verbose=0)[0]
    idx = int(np.argmax(probs))
    return {
        "class"      : CLASS_NAMES[idx],
        "description": CLASS_DESC[idx],
        "confidence" : f"{probs[idx]*100:.1f}%",
        "probabilities": {CLASS_NAMES[k]: f"{probs[k]*100:.1f}%" for k in range(3)},
    }


# Tests with real SDSS objects
test_objects = [
    ("Typical spiral galaxy",
     dict(u=19.47, g=17.89, r=17.12, i=16.76, z=16.52, redshift=0.089)),
    ("G-type star (similar to the Sun)",
     dict(u=18.82, g=17.41, r=16.93, i=16.72, z=16.59, redshift=0.00012)),
    ("High-redshift quasar",
     dict(u=19.14, g=18.97, r=18.76, i=18.59, z=18.32, redshift=1.847)),
]

for name, params in test_objects:
    res = classify_celestial_object(**params)
    print(f"\nObject      : {name}")
    print(f"Class       : {res['class']}")
    print(f"Confidence  : {res['confidence']}")
    print(f"Probabilities: {res['probabilities']}")
    print(f"{res['description']}")


---
## 14. Saving and reproducibility

We save all artefacts: the trained model, the scaler, and experiment metadata.
This is the foundation of any **MLOps** practice (operationalisation of ML models).


In [ ]:
os.makedirs("models", exist_ok=True)

# Keras model (recommended native format)
model.save("models/cosmic_census_net.keras")

# Scaler — required to normalize new input data
with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Experiment metadata (traceability — MLOps principle)
metadata = {
    "project"      : "Cosmic Census",
    "dataset"      : "SDSS DR17",
    "n_train"      : int(X_tr.shape[0]),
    "n_test"       : int(X_te.shape[0]),
    "features"     : FEATURES,
    "classes"      : ["GALAXY", "STAR", "QSO"],
    "metrics"      : {
        "random_forest" : round(float((y_pred_rf == y_te).mean()), 4),
        "neural_network": round(float(acc_te), 4),
        "spark_mllib"   : round(float(acc_sp), 4),
    },
    "architecture" : "MLP Dense 256->128->64->3 (BatchNorm+Dropout)",
    "date"         : datetime.now().isoformat()
}
with open("models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Artefacts saved to ./models/")
print(json.dumps(metadata, indent=2))


---
## 15. Conclusion and next steps

### What we built

| Stage | Technology | Result |
|---|---|---|
| Data ingestion | Python + requests | 100k real SDSS objects |
| Data Lake | Parquet Snappy | ~39% smaller than CSV |
| Distributed processing | PySpark DataFrames | Cleaning + features in Spark |
| Manual MapReduce | PySpark RDD | Fundamental pattern demonstrated |
| EDA and visualization | Pandas + Plotly | 5 interactive charts |
| Classic ML | Scikit-learn Random Forest | ~98% accuracy |
| Deep Learning | TensorFlow/Keras MLP | ~97% accuracy |
| Distributed pipeline | Spark MLlib | Scalable to terabytes |
| Deployment | Documented function + artefacts | Production-ready |

### Technical conclusion

For this dataset, **Random Forest slightly outperformed the neural network** (~98% vs ~97%).
This is common with structured tabular data — tree-based models tend to be competitive
or superior to neural networks when the number of features is small.
The neural network would show a greater advantage with more features or raw data (FITS images).

### Next steps to evolve the project

1. **Larger data** — download the full dataset via SDSS TAP/SQL (500+ million objects)
2. **CNNs on images** — use the actual FITS images with convolutional neural networks
3. **Streaming** — real-time ingestion with Spark Structured Streaming
4. **Deployment** — package as a REST API with FastAPI and containerize with Docker
5. **Cloud** — run the pipeline on Google Dataproc or AWS EMR

### Scientific references

- Abazajian et al. (2009). *The Seventh Data Release of the SDSS*. ApJS, 182, 543
- York et al. (2000). *The Sloan Digital Sky Survey: Technical Summary*. AJ, 120, 1579
- Apache Spark MLlib: https://spark.apache.org/mllib/
- SDSS SkyServer: https://skyserver.sdss.org

---
*Cosmic Census — Celestial Object Classification with Big Data*
*Computer Engineering | SDSS Data Release 17*
